In [1]:
%pip uninstall -y tensorflow keras keras-vggface keras-applications mtcnn
%pip install tensorflow
%pip install keras_applications==1.0.8
%pip install mtcnn==0.1.0

# Install keras-vggface directly from a patch repository that fixes the modern TensorFlow compatibility issues
%pip install git+https://github.com/rcmalli/keras-vggface.git

Found existing installation: Keras 2.4.3
Uninstalling Keras-2.4.3:
  Successfully uninstalled Keras-2.4.3
Note: you may need to restart the kernel to use updated packages.


  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.36.2-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 2.1/351.2 MB 11.0 MB/s eta 0:00:32
    --------------------------------------- 4.7/351.2 MB 11.5 MB/s eta 0:00:31
    --------------------------------------- 7.1/351.2 MB 11.6 MB/s eta 0:00:30
   - -------------------------------------- 9.7/351.2 MB 11.6 MB/s eta 0:00:30
   - -------------------------------------- 12.1/351.2 MB 11.6 MB/s eta 0:00:30
   - -------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires protobuf<7,>=3.20, but you have protobuf 7.36.2 which is incompatible.



   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ----------------------- ---------------- 1.3/2.3 MB 6.5 MB/s eta 0:00:01
   ------------------------------------- -- 2.1/2.3 MB 8.1 MB/s eta 0:00:01
   ------------------------------------- -- 2.1/2.3 MB 8.1 MB/s eta 0:00:01
   ---------------------------------------- 2.3/2.3 MB 3.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
  Cloning https://github.com/rcmalli/keras-vggface.git to c:\users\vicky\appdata\local\temp\pip-req-build-lqyye6_1
  Resolved https://github.com/rcmalli/keras-vggface.git to commit bee35376e76e35d00aeec503f2f242611a97b38a
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for keras_vggface: filename=keras_vggface-0.6-py3-none-any.whl size=8376 sha256=d2f256255750ba23dfbbb7a361ff211d9717c65cb854c05d2bc0704ae80f64d2
  Stored in directory: C:\Users\VICKY\AppData\Local\Temp\pip-ephem-wheel-ca

  Running command git clone --filter=blob:none --quiet https://github.com/rcmalli/keras-vggface.git 'C:\Users\VICKY\AppData\Local\Temp\pip-req-build-lqyye6_1'
  DEPRECATION: Building 'keras_vggface' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'keras_vggface'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [3]:
%pip install "tensorflow<2.16"

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement tensorflow<2.16 (from versions: 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0, 2.22.0rc0)
ERROR: No matching distribution found for tensorflow<2.16


In [3]:
%pip install tf-keras

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 10.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import sys
import pickle
import numpy as np
from pathlib import Path

# CHANGED: Use the standard text progress bar so the UI widget doesn't crash
from tqdm import tqdm 

from tensorflow.keras.preprocessing import image
import tensorflow as tf

sys.modules['keras.utils.layer_utils'] = tf.keras.utils
sys.modules['keras.utils.data_utils'] = tf.keras.utils
sys.modules['keras.backend'] = tf.keras.backend
sys.modules['keras.engine.topology'] = tf.keras.layers

from keras_vggface.utils import preprocess_input
from keras_vggface.vggface import VGGFace

PROJECT_ROOT = Path(r"E:\Python Projects\Celebrity_Face_matcher")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURE_DIR = PROJECT_ROOT / "data" / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Collect filenames
filenames = []
for actor_dir in sorted(PROCESSED_DIR.iterdir()):
    if actor_dir.is_dir():
        for img_path in sorted(actor_dir.iterdir()):
            if img_path.suffix.lower() == '.jpg':
                filenames.append(str(img_path))

print(f"Total curated images for feature extraction: {len(filenames)}")

# 2. Load Model
print("Loading VGGFace ResNet50 model...")
model = VGGFace(model='resnet50', include_top=False, input_shape=(224, 224, 3), pooling='avg')

# 3. Extraction Function
def feature_extractor(img_path, model):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    expanded_img = np.expand_dims(img_array, axis=0)
    preprocessed_img = preprocess_input(expanded_img)
    result = model.predict(preprocessed_img, verbose=0).flatten()
    return result

# 4. Process and Save
features = []
for file in tqdm(filenames, desc="Extracting Embeddings"):
    features.append(feature_extractor(file, model))

filenames_path = FEATURE_DIR / 'filenames.pkl'
embeddings_path = FEATURE_DIR / 'embedding.pkl'

pickle.dump(filenames, open(filenames_path, 'wb'))
pickle.dump(features, open(embeddings_path, 'wb'))

print(f"Extraction complete! Saved to:\n- {filenames_path}\n- {embeddings_path}")

Total curated images for feature extraction: 15842
Loading VGGFace ResNet50 model...




Extracting Embeddings: 100%|██████████| 15842/15842 [44:44<00:00,  5.90it/s] 


Extraction complete! Saved to:
- E:\Python Projects\Celebrity_Face_matcher\data\features\filenames.pkl
- E:\Python Projects\Celebrity_Face_matcher\data\features\embedding.pkl
